# Lab 6 essentials: questions in English, numbers you can defend

**Time: about 50 minutes. Click Run all, then read each step while it runs. Do Lab 5 essentials first.**

**What you will do.** A manager asks "How many orders were placed in May 2024?" and wants one number.
You let a language model write the SQL for FreshCart's warehouse, check its numbers against reviewed
answers, add the business rules it did not know, and score it on 16 questions. Then you take the SQL
away from the model altogether: a **semantic layer** holds each business number, written once, and
the model only picks which one. You finish by turning a number into a sentence and a chart. Along the
way you will find two problems that no error message tells you about.

**The libraries, and what each one does.** Nothing else is used: no lab-specific code.

| Library | What it does here | Step | Docs |
|---|---|---|---|
| `langchain-community` | `SQLDatabase`: reads the warehouse's tables, columns and sample rows, ready for a prompt | 1 | [python.langchain.com](https://python.langchain.com/docs/integrations/tools/sql_database/) |
| `sqlite3` (built into Python) | runs the SQL on the warehouse, a single file, opened read-only | 1 | [docs.python.org](https://docs.python.org/3/library/sqlite3.html) |
| `langchain-core` | `ChatPromptTemplate` (instructions with blanks), `StrOutputParser` (the reply as plain text), and the `|` that chains steps | 2 | [python.langchain.com](https://python.langchain.com/docs/concepts/lcel/) |
| `langchain-openai` | `ChatOpenAI`: talks to the Databricks model, as in Lab 5 | 2 | [python.langchain.com](https://python.langchain.com/docs/integrations/chat/openai/) |
| `pydantic` | describes the exact shape of the plan the model fills in | 8 | [docs.pydantic.dev](https://docs.pydantic.dev/) |
| `matplotlib` (already on Databricks) | draws the charts | 9 | [matplotlib.org](https://matplotlib.org/stable/) |

To see every setting a library offers, run the **Look it up yourself** cell at the end, or type
`help(name)` in any cell.

**Five words used throughout.**

- **NL2SQL** (natural language to SQL): a model turns a question in English into a SQL query.
- The **schema** is the list of tables and their columns. It is all the model knows about the
  warehouse unless you tell it more.
- **Business rules** are what the company means by its words: which orders count as revenue, which
  customers are test accounts. They are not in the schema.
- **Execution accuracy**: a generated query is right if it returns the same rows as the reviewed
  query. The SQL text can differ.
- A **semantic layer** is a list of business numbers, each written once with its rules, in one place.
  Questions are matched to a number in the list instead of being turned into new SQL.

**Cost.** The model is served by Databricks and billed per use: this notebook makes about 75 short
calls, a few cents in total.

## Step 0. Install the libraries

**What this does.** Installs the libraries from the table above, then restarts Python. It takes about
one minute. The restart is expected.

In [ ]:
# The SQL helper and the model connector. pydantic and matplotlib are already on Databricks.
%pip install -q langchain-core langchain-community langchain-openai

In [ ]:
# Restart Python so the libraries installed above are the ones that get loaded.
dbutils.library.restartPython()

In [ ]:
# warnings controls which notices are shown.
import warnings

# Hide notices that are not errors, such as messages about packages being renamed.
warnings.filterwarnings("ignore")
print("ready")

## Step 1. The warehouse, and what the model will be told about it

**What this does.** Copies the warehouse, `freshcart.db`, out of the course zip file. `SQLDatabase`
then writes the **schema** as text: for each table, the command that created it plus two sample rows.
That text is what the model gets in Step 2.

`run` is our own function for running SQL. It opens the warehouse **read-only**, so no query can
change it (Lab 5, Problem 2).

**What you should see.** The eight table names, then the schema text for `orders_v2` and
`dim_customer`. Notice the column `is_internal` in `dim_customer`: nothing in the schema says what
it is for.

In [ ]:
# The course files: the documents, the warehouse and the test questions, from this lab's GitHub
# repository. They are downloaded once and kept in one zip file, PACK_ZIP, that later steps read.
import io, pathlib, urllib.request, zipfile

PACK_ZIP = "/tmp/freshcart_course_files.zip"
if not pathlib.Path(PACK_ZIP).exists():
    repo = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://github.com/mogupta84/ai-engineer-data-labs/archive/refs/heads/main.zip", timeout=120).read()))
    with zipfile.ZipFile(PACK_ZIP, "w") as out:
        for name in repo.namelist():
            inside = name.split("/", 1)[1] if "/" in name else ""     # drop the repository folder name
            if inside.startswith("data/") and not name.endswith("/"):
                out.writestr(inside, repo.read(name))
print("course files ready:", PACK_ZIP)

In [ ]:
# zipfile reads the course zip file; pathlib handles file paths; sqlite3 opens the warehouse.
import zipfile, pathlib, sqlite3
from langchain_community.utilities import SQLDatabase

# Copy the warehouse into a folder of our own.
PACK = PACK_ZIP
HOME = pathlib.Path("/tmp/lab06_essentials")
HOME.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACK) as z:
    (HOME / "freshcart.db").write_bytes(z.read("data/freshcart.db"))
DB = HOME / "freshcart.db"

# SQLDatabase reads the tables; sample_rows_in_table_info=2 adds two example rows per table.
warehouse = SQLDatabase.from_uri(f"sqlite:///{DB}", sample_rows_in_table_info=2)
SCHEMA = warehouse.get_table_info()

# run(sql): the rows a query returns. mode=ro opens the file read-only.
def run(sql):
    connection = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
    try:
        return connection.execute(sql).fetchall()
    finally:
        connection.close()

print(warehouse.get_usable_table_names(), "\n")
print(warehouse.get_table_info(["orders_v2", "dim_customer"]))

## Step 2. The model writes the SQL

**What this does.** Builds the SQL writer as a LangChain chain of three steps joined by `|`:

1. `ChatPromptTemplate`: the instructions, with the schema and the question filled into blanks;
2. the model, `databricks-meta-llama-3-3-70b-instruct`, connected as in Lab 5;
3. `StrOutputParser`: takes the text out of the model's reply.

A small function then removes the ```` ```sql ```` marks a model sometimes wraps around its query.
The writer is asked "How many orders were placed in May 2024?", and the query is run. Beside it, the
**reviewed query**: the one a FreshCart analyst checked by hand.

**What you should see.** A tidy query, and two numbers: **491** from the model's query and **450**
from the reviewed one. The model's query runs without an error, and 491 looks just as believable as
450. Only the comparison shows something is wrong. In one test run the model picked the old
`orders` table instead and got **306**: a second trap of the same kind, which the rules in Step 4 name.

In [ ]:
from databricks.sdk import WorkspaceClient
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.rate_limiters import InMemoryRateLimiter

# Sign in as you; the token goes straight into the connector and is never printed (as in Lab 5).
workspace = WorkspaceClient()
token = workspace.config.authenticate()["Authorization"].split(" ", 1)[1]
llm = ChatOpenAI(model="databricks-meta-llama-3-3-70b-instruct",
                 base_url=workspace.config.host.rstrip("/") + "/serving-endpoints",
                 api_key=token, temperature=0, max_retries=5,
                 # The served model allows only a few calls a second for the whole workspace: send at most 2.
                 rate_limiter=InMemoryRateLimiter(requests_per_second=2))

# The instructions. {schema}, {rules} and {question} are blanks, filled in for every question.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You write one SQLite query that answers the question. Reply with the SQL only, no explanation.\n\n"
               "Tables:\n{schema}\n\n{rules}"),
    ("human", "{question}"),
])

# Fill the prompt, call the model, take out the text: three steps in one chain.
sql_writer = prompt | llm | StrOutputParser()

# Remove ```sql and ``` if the model wrapped its query in them.
def clean(sql):
    return sql.strip().removeprefix("```sql").removeprefix("```").removesuffix("```").strip()

# write_sql(question, rules): the model's query, ready to run. With no rules, the blank stays empty.
def write_sql(question, rules=""):
    return clean(sql_writer.invoke({"schema": SCHEMA, "rules": rules, "question": question}))

question = "How many orders were placed in May 2024?"
REVIEWED = ("SELECT COUNT(*) FROM orders_v2 o JOIN dim_customer c ON c.customer_id = o.customer_id "
            "WHERE c.is_internal = 0 AND o.order_date BETWEEN '2024-05-01' AND '2024-05-31'")
model_sql = write_sql(question)
print("model's query :", model_sql)
print("model's number:", run(model_sql)[0][0])
print("reviewed      :", run(REVIEWED)[0][0])

### Step 3. Check: where do the 41 extra orders come from?

**What this does.** The reviewed query has one condition the model's does not:
`c.is_internal = 0`. This cell looks at the customers with `is_internal = 1` and counts their orders
in May 2024.

**What you should see.** Two customers, `customer_id` 1 and 2, FreshCart's **test accounts**, which
the engineers use to place fake orders. They placed **41** orders in May 2024, and 491 minus 41 is
450. The model could not have known: the schema has a column called `is_internal`, but nothing says
that it marks test accounts or that they must be left out. That is a **business rule**, and it lives
in people's heads, not in the database.

In [ ]:
# Who are the internal customers, and how many May orders did they place?
print(run("SELECT customer_id, name, is_internal FROM dim_customer WHERE is_internal = 1"))
print("their orders in May 2024:",
      run("SELECT COUNT(*) FROM orders_v2 o JOIN dim_customer c ON c.customer_id = o.customer_id "
          "WHERE c.is_internal = 1 AND o.order_date BETWEEN '2024-05-01' AND '2024-05-31'")[0][0])

## Step 4. Tell the model the business rules

**What this does.** Writes FreshCart's business rules as eight plain lines and fills them into the
`{rules}` blank of the same prompt. Nothing else changes. Then the May question is asked again.

**What you should see.** Still **491**, in both test runs. The query is the same as before: the
model read the rule "exclude test accounts" and did not apply it to a simple count. A rule in the
prompt is a request, not a guarantee. Step 5 tries a different way of telling it.

In [ ]:
# What FreshCart means by its words. None of this is in the schema.
RULES = """Business rules:
- exclude test accounts: always join dim_customer and keep only is_internal = 0
- orders is an old copy; always use orders_v2
- revenue = SUM(orders_v2.order_value) where status = 'delivered'
- never join order_items when adding up order_value: each order would be counted once per item
- an active customer placed at least one order in orders_v2 in the previous 90 days
- dim_customer.status is old and meaningless; never use it
- daily_customer_snapshot holds running totals per month-end; never add it up across dates
- last month = the previous calendar month"""

model_sql = write_sql(question, RULES)
print("model's query :", model_sql)
print("model's number:", run(model_sql)[0][0], "  reviewed:", run(REVIEWED)[0][0])

## Step 5. Show the model four worked examples

**What this does.** Adds four **worked examples** to the prompt: a question and the reviewed query
that answers it. Giving examples like this is called **few-shot** prompting. The examples follow the
rules, so the model can copy their shape: which tables to start from, the `dim_customer` join, and how
to write a date range. The first example is an order count for March, a different month from the one
we ask about.

**What you should see.** The query now joins `dim_customer` with `is_internal = 0`, and the number is
**450**, matching the reviewed answer. One example did what eight lines of rules did not.

In [ ]:
# Four questions with reviewed queries. None of them is one of the 16 test questions.
EXAMPLES = """Worked examples:
Q: How many orders were placed in March 2024?
SQL: SELECT COUNT(*) FROM orders_v2 o JOIN dim_customer c ON c.customer_id = o.customer_id WHERE c.is_internal = 0 AND o.order_date BETWEEN '2024-03-01' AND '2024-03-31'
Q: What was revenue in Q1 2024?
SQL: SELECT SUM(o.order_value) FROM orders_v2 o JOIN dim_customer c ON c.customer_id = o.customer_id WHERE c.is_internal = 0 AND o.status = 'delivered' AND o.order_date BETWEEN '2024-01-01' AND '2024-03-31'
Q: How many active customers are there as of 2024-06-30?
SQL: SELECT COUNT(DISTINCT o.customer_id) FROM orders_v2 o JOIN dim_customer c ON c.customer_id = o.customer_id WHERE c.is_internal = 0 AND o.order_date BETWEEN date('2024-06-30', '-90 day') AND '2024-06-30'
Q: How many refund claims did customer 91 make in October 2024?
SQL: SELECT COUNT(*) FROM fact_refund_claim WHERE customer_id = 91 AND claim_date BETWEEN '2024-10-01' AND '2024-10-31'"""

# The rules and the examples go into the same {rules} blank.
model_sql = write_sql(question, RULES + "\n\n" + EXAMPLES)
print("model's query :", model_sql)
print("model's number:", run(model_sql)[0][0], "  reviewed:", run(REVIEWED)[0][0])

## Step 6. Score it on 16 questions, three ways

**What this does.** Reads 16 test questions, each with a reviewed query, from the course zip file.
For every question the model writes a query three times: with the schema only, with the rules, and
with the rules and the examples. Each query is scored
by **execution accuracy**: it is right if it returns the same rows as the reviewed query, with numbers
rounded to 2 decimal places. For the two "which" questions only the top answer counts. A query that
fails to run counts as wrong.

**What you should see.** Close to these, which can move by one or two between runs:

- **schema only: 7 or 8 of 16**
- **with the rules: 10 or 11 of 16**
- **with the rules and examples: 14 of 16**

Read the table. The rules fixed questions about revenue and customers, where test accounts and
undelivered orders matter, and the examples fixed the order counts, "active customers" and the average order value. Two fail
every time: the average delivery time (N10) and "last month" (N16), which does not say which month.
14 of 16 sounds good, but you only know which 14 because a person wrote 16 reviewed answers. Step 8
shows a different way to handle that.

In [ ]:
# json reads the test questions.
import json

with zipfile.ZipFile(PACK) as z:
    tests = json.loads(z.read("data/golden/nl2sql_questions.json"))["questions"]

# rows_of(sql): the rows, with numbers rounded so 450.0 and 450 match. None if the query fails.
def rows_of(sql, first_column=False):
    try:
        rows = run(sql)
    except Exception:
        return None
    # For "which" questions only the top row's first column counts: extra rows and columns are fine.
    if first_column:
        rows = [r[:1] for r in rows[:1]]
    return sorted(tuple(round(v, 2) if isinstance(v, float) else v for v in r) for r in rows)

# is_right: the model's rows equal the reviewed rows.
def is_right(sql, test):
    first_column = test.get("compare") == "first_column"
    mine = rows_of(sql, first_column)
    return mine is not None and mine == rows_of(test["ref_sql"], first_column)

# The three ways of prompting, from least to most help.
WAYS = {"schema only": "", "rules": RULES, "rules + examples": RULES + "\n\n" + EXAMPLES}
mark = {True: "right", False: "wrong"}
totals = {way: 0 for way in WAYS}
for test in tests:
    results = {way: is_right(write_sql(test["question"], extra), test) for way, extra in WAYS.items()}
    for way, ok in results.items():
        totals[way] += ok
    print(test["id"], "  ".join(f"{way}: {mark[ok]:5}" for way, ok in results.items()), " ", test["question"])
print("\n" + "    ".join(f"{way}: {total}/16" for way, total in totals.items()))

## Step 7. Problem 2: one phrase, two numbers

**What this does.** "Refund rate for March 2024" sounds like one number. This cell works it out the
two ways a team could reasonably mean it:

- **claims per order**: number of refund claims divided by number of orders;
- **orders with a claim**: number of orders that had at least one claim, divided by number of orders.

**What you should see.** **0.0452** and **0.0409**. Both are honest, and both would pass a review on
their own. They differ because some orders have more than one claim. If finance uses one and support
uses the other, the two teams bring different numbers to the same meeting, and the business rules in
Step 4 do not say which is right. Writing more rules into a prompt cannot settle this: someone has to
decide, once, and the decision has to live in one place.

In [ ]:
# The same orders and claims for March 2024, test accounts left out.
march = ("FROM orders_v2 o JOIN dim_customer c ON c.customer_id = o.customer_id "
         "LEFT JOIN fact_refund_claim f ON f.order_id = o.order_id "
         "WHERE c.is_internal = 0 AND o.order_date BETWEEN '2024-03-01' AND '2024-03-31'")
# Claims divided by orders.
claims_per_order = run(f"SELECT 1.0 * COUNT(DISTINCT f.claim_id) / COUNT(DISTINCT o.order_id) {march}")[0][0]
# Orders that had a claim, divided by orders.
orders_with_claim = run(f"SELECT 1.0 * COUNT(DISTINCT f.order_id) / COUNT(DISTINCT o.order_id) {march}")[0][0]
print(f"refund rate, claims per order   : {claims_per_order:.4f}")
print(f"refund rate, orders with a claim: {orders_with_claim:.4f}")

## Step 8. The semantic layer: the model picks, the code writes the SQL

**What this does.** Writes the semantic layer: four business numbers, each with its formula, its
tables and its rules, written once. For refund rate, the decision from Step 7 is made here: claims per
order. Three **breakdowns** are allowed: by city, by region, by month.

The model no longer writes SQL. It fills in a short form, a **plan**: which number, which breakdown,
and which dates. `pydantic` describes the plan's exact shape, and `"none"` is one of the choices. The
plan is given to the model as its only **tool**, exactly as in Lab 5, so the model's tool call is the
filled-in form. `PydanticToolsParser` reads that tool call back as a `Plan`. Code then writes the SQL
from the plan, so a rule can never be forgotten.

The same 16 questions go through the layer. When the plan says `"none"`, the layer **refuses**: it
has no number for that question and will not make one up.

**What you should see.** In both test runs:

- the layer **answers 6** questions and **refuses 10**, and **4 of the 6** are right;
- **N07**, "refund claims **approved** in March": the plan has no place for a status, so the model
  quietly dropped the word "approved" and returned all claims;
- **N16**, "orders **last month**": the model guessed which month, although the instructions said to
  refuse.

That is the trade. The SQL writer in Step 6 answered all 16 and was right on 14, but you could only
tell which 14 because someone wrote the reviewed answers. The layer answers fewer, and when it
answers, the formula and its rules are always right. Its two mistakes are in **reading the
question**, not in the SQL, and they are easy to spot because the plan is printed beside the answer.
Adding a number to the layer, such as average order value, is how it grows.

**Try this.** Delete everything after "answers the question exactly" in the planner's instructions and
run this cell again. A test run answered 9 and got only 4 right: without being told when to
refuse, the model forces a question into the nearest number.

In [ ]:
# Literal lists the allowed values; BaseModel describes the shape of the plan.
from typing import Literal
from pydantic import BaseModel, Field

# Each business number, written once: formula, tables, rules, and which column holds its date.
METRICS = {
    "revenue": {"formula": "SUM(o.order_value)",
                "tables": "orders_v2 o JOIN dim_customer c ON c.customer_id = o.customer_id",
                "rules": ["c.is_internal = 0", "o.status = 'delivered'"], "date": "o.order_date"},
    "order_count": {"formula": "COUNT(DISTINCT o.order_id)",
                    "tables": "orders_v2 o JOIN dim_customer c ON c.customer_id = o.customer_id",
                    "rules": ["c.is_internal = 0"], "date": "o.order_date"},
    "refund_claims": {"formula": "COUNT(DISTINCT f.claim_id)",
                      "tables": "fact_refund_claim f JOIN dim_customer c ON c.customer_id = f.customer_id",
                      "rules": ["c.is_internal = 0"], "date": "f.claim_date"},
    # Refund rate means claims per order: decided once, here (Step 7).
    "refund_rate": {"formula": "1.0 * COUNT(DISTINCT f.claim_id) / COUNT(DISTINCT o.order_id)",
                    "tables": "orders_v2 o JOIN dim_customer c ON c.customer_id = o.customer_id "
                              "LEFT JOIN fact_refund_claim f ON f.order_id = o.order_id",
                    "rules": ["c.is_internal = 0"], "date": "o.order_date"},
}
# The allowed breakdowns. "month" is worked out from each number's own date column.
BREAKDOWNS = {"city": "c.city", "region": "c.region"}

# The plan the model must fill in. The descriptions are what the model reads.
class Plan(BaseModel):
    metric: Literal["revenue", "order_count", "refund_claims", "refund_rate", "none"] = Field(
        description="the business number that answers the question exactly; 'none' if no number fits exactly")
    breakdown: Literal["city", "region", "month", "none"] = Field(description="split the number by this, or 'none'")
    start_date: str = Field(description="first day, YYYY-MM-DD")
    end_date: str = Field(description="last day, YYYY-MM-DD, included")

from langchain_core.output_parsers import PydanticToolsParser

# The instructions, the model with Plan as its only tool, and a parser that turns the tool call into a Plan.
plan_prompt = ChatPromptTemplate.from_messages([
    ("system", "Turn the question into a plan by calling the Plan tool. Use metric 'none' when none of the "
               "listed business numbers answers the question exactly: when the question adds a condition the "
               "plan cannot hold (a status, one customer, an average), counts something else, or does not "
               "state its dates exactly (such as 'last month'). Use a breakdown only when the question asks "
               "for a split or for 'which' city or region."),
    ("human", "{question}"),
])
planner = plan_prompt | llm.bind_tools([Plan]) | PydanticToolsParser(tools=[Plan], first_tool_only=True)

# sql_for(plan): the layer writes the query. Every rule of the metric is always added.
def sql_for(plan):
    m = METRICS[plan.metric]
    where = " AND ".join(m["rules"] + [f"{m['date']} BETWEEN '{plan.start_date}' AND '{plan.end_date}'"])
    if plan.breakdown == "none":
        return f"SELECT {m['formula']} FROM {m['tables']} WHERE {where}"
    group = f"substr({m['date']}, 1, 7)" if plan.breakdown == "month" else BREAKDOWNS[plan.breakdown]
    return f"SELECT {group}, {m['formula']} FROM {m['tables']} WHERE {where} GROUP BY 1 ORDER BY 2 DESC"

answered, right = 0, 0
for test in tests:
    plan = planner.invoke({"question": test["question"]})
    # No plan at all counts as a refusal too.
    if plan is None or plan.metric == "none":
        print(f"{test['id']}  refused                            {test['question']}")
        continue
    answered += 1
    ok = is_right(sql_for(plan), test)
    right += ok
    print(f"{test['id']}  {plan.metric:13} {plan.breakdown:6} {'right' if ok else 'WRONG':5}  {test['question']}")
print(f"\nthe layer answered {answered} of 16, refused {16 - answered}; right: {right} of {answered}")

## Step 9. From a number to a sentence and a chart

**What this does.** Turns layer numbers into what a manager reads. Two rules, both in code:

1. **The sentence is built from the number, not written by the model.** It states the number, the
   period, the comparison and the scope, and never a cause, because nothing in the data says why a
   number moved. In Lab 5 the model added a dollar sign of its own; here there is no room for that.
2. **The chart type follows the shape of the data.** Places with no natural order become a bar chart,
   sorted. A number over time becomes a line, read left to right.

**What you should see.** One sentence: revenue was 531,043.03 in March 2024, up 1.7% from 522,100.56
in February 2024, for delivered orders excluding test accounts. Then two charts: revenue by city in
Q2 2024 as bars, and orders by month in 2024 as a line.

In [ ]:
import matplotlib.pyplot as plt

# One number from the layer, for a plan written by hand.
def number(metric, start, end):
    return run(sql_for(Plan(metric=metric, breakdown="none", start_date=start, end_date=end)))[0][0]

march = number("revenue", "2024-03-01", "2024-03-31")
february = number("revenue", "2024-02-01", "2024-02-29")
# up or down, and by how much, worked out in code.
change = (march - february) / february * 100
direction = "up" if change > 0 else "down"
print(f"Revenue was {march:,.2f} in March 2024, {direction} {abs(change):.1f}% from {february:,.2f} "
      f"in February 2024. Scope: delivered orders, excluding test accounts.")

# chart(plan, title): draws a layer result, choosing the chart from the breakdown.
def chart(plan, title, axis):
    rows = run(sql_for(plan))
    labels, values = [r[0] for r in rows], [r[1] for r in rows]
    if plan.breakdown == "month":
        # Over time: a line, in date order.
        pairs = sorted(zip(labels, values))
        axis.plot([p[0] for p in pairs], [p[1] for p in pairs], marker="o")
        axis.tick_params(axis="x", rotation=45)
    else:
        # Places: bars, already sorted from biggest to smallest by the layer.
        axis.bar(labels, values)
    axis.set_title(title)

figure, (left, right_axis) = plt.subplots(1, 2, figsize=(12, 4))
chart(Plan(metric="revenue", breakdown="city", start_date="2024-04-01", end_date="2024-06-30"),
      "Revenue by city, Q2 2024 (bar: no natural order)", left)
chart(Plan(metric="order_count", breakdown="month", start_date="2024-01-01", end_date="2024-12-31"),
      "Orders by month, 2024 (line: over time)", right_axis)
plt.tight_layout()
plt.show()

## What you found

1. **A model's SQL can be wrong with no error** (Steps 2 and 3): 491 instead of 450, because it
   counted FreshCart's test accounts. Only a reviewed answer showed it.
2. **Rules and examples help, but do not finish the job** (Steps 4 to 6): the model ignored a rule
   it had been given, an example fixed it, and the score rose from about 7 to 14 of 16.
3. **Some words have two honest meanings** (Step 7): refund rate is 0.0452 or 0.0409. A prompt cannot
   decide that; a person has to, once.
4. **A semantic layer answers fewer questions, and follows the rules every time** (Step 8): 6
   answered, 4 right, 10 refused. Its two mistakes were in reading the question, and the plan
   printed beside each answer shows them.
5. **Write the sentence in code** (Step 9): number, period, comparison, scope, and no cause.

**Where this goes next.** In a real warehouse the semantic layer usually lives in a tool such as dbt
or Databricks metric views, so every dashboard and every agent reads the same definitions. The
`get_metric` tool from Lab 5 is exactly this idea, offered to a model as a tool.

## Look it up yourself: every setting the libraries offer

**What this does.** For the main things this lab calls, the cell prints every parameter (read from the
installed library), the values this lab passed, and one setting worth trying.

**What you should see.** Four short blocks.

In [ ]:
# inspect reads the list of parameters a function or class accepts; textwrap folds long lines.
import inspect, textwrap

# parameters(thing): the parameters `thing` accepts, as text such as "temperature=None".
def parameters(thing):
    # Some classes list their settings as "fields", not on __init__.
    if hasattr(thing, "model_fields"):
        return [n if f.is_required() else f"{n}={f.get_default(call_default_factory=True)!r}"
                for n, f in thing.model_fields.items()]
    try:
        sig = inspect.signature(thing)
    except (TypeError, ValueError):
        return []
    # Skip "self", "cls", and the *args / **kwargs catch-alls, which are not named settings.
    return [p.name if p.default is p.empty else f"{p.name}={p.default!r}" for p in sig.parameters.values()
            if p.name not in ("self", "cls") and p.kind not in (p.VAR_POSITIONAL, p.VAR_KEYWORD)]

# look_up prints the three lines for one function or class.
def look_up(name, thing, this_lab_passed, worth_a_look):
    print(name)
    print(textwrap.fill("every parameter : " + ", ".join(parameters(thing)), 110, subsequent_indent=" " * 18))
    print("this lab passed :", this_lab_passed)
    print("worth a look    :", worth_a_look, "\n")

look_up("SQLDatabase.from_uri", SQLDatabase.from_uri, 'f"sqlite:///{DB}", sample_rows_in_table_info=2',
        "include_tables: send only the tables a question needs, as real warehouses have thousands")
look_up("SQLDatabase.get_table_info", SQLDatabase.get_table_info, "nothing (every table), then two table names",
        "table_names: the same list, chosen per question")
look_up("PydanticToolsParser", PydanticToolsParser, "tools=[Plan], first_tool_only=True",
        "first_tool_only=False: a list of every plan, for a question that needs two numbers")
look_up("Field", Field, "description=...",
        "the descriptions: they are the planner's only instructions, like a tool description in Lab 5")

## Your notes

Double click to type: which question from Step 8 you would add to the semantic layer next, and the
formula, tables and rules you would write for it.